# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mariamemad975/FlyRank_ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — “What Predicts Health?”

The Random Forest reports Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the strongest predictors of Health Score. However, these variables are direct components of the Health Score formula itself: Impressions (30), Position (30), CTR (20), and Scroll Depth (20).

Methodological question: Does predicting a composite score from its own ingredients provide information beyond confirming the construction formula? The feature importance may be better interpreted as evidence of the score's construction weights rather than independent predictive importance.

Finding 2 — “What Predicts Growth?”

The Logistic Regression reports 71% holdout accuracy, with Content Age, Days Since Update, and Days Visible as the strongest signals. Trend Direction, however, is defined from the change between recent and previous impression periods.

Methodological question: Were any input features derived from or closely aligned with the same information used to define the label, making them near-restatements of the target? Also, was the holdout split random or time-aware? A random split could allow similar observations from the same client to appear in both training and test sets, which may make performance look stronger than generalization to unseen clients.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model evaluated machine-learning approaches for detecting CTR/engagement decline. For this validation audit, I use the Random Forest as the main model so that the comparison between the original random split and the client-grouped evaluation is based on the same model and feature set.

The original random split can place pages from the same client in both training and test sets. Because pages from one client may share characteristics, this can produce an optimistic estimate of generalization.

I therefore use GroupKFold with client_id as the grouping variable. This keeps all pages from each client within either the training or test portion of a fold and provides a more realistic estimate of performance on unseen clients.

The dataset does not expose a suitable timestamp for a fully time-aware split, so client-grouped validation is used instead.

In [59]:
%pip install -q duckdb pandas scikit-learn matplotlib

In [60]:
from pathlib import Path

repo_path = Path("/content/FlyRank_ML")

if not repo_path.exists():
    !git clone https://github.com/mariamemad975/FlyRank_ML.git /content/FlyRank_ML

%cd /content/FlyRank_ML

print("Working directory:", Path.cwd())

/content/FlyRank_ML
Working directory: /content/FlyRank_ML


In [61]:
from pathlib import Path

data_path = Path("data/raw/content_refresh_anonymized.csv")

print("Dataset path:", data_path)
print("Dataset exists:", data_path.exists())

assert data_path.exists(), f"Dataset not found: {data_path}"

Dataset path: data/raw/content_refresh_anonymized.csv
Dataset exists: True


In [62]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    average_precision_score)

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("Number of columns:", len(df.columns))

Dataset shape: (30000, 44)
Number of columns: 44


In [63]:
df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int))

print("Label distribution:")
print(df["is_declining_label"].value_counts())

print("\nLabel proportions:")
print(df["is_declining_label"].value_counts(normalize=True).round(3))

Label distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Label proportions:
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


In [64]:
drop_cols = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"]

feature_cols = [
    column for column in df.columns
    if column not in drop_cols]

numeric_features = (
    df[feature_cols]
    .select_dtypes(include=[np.number])
    .columns
    .tolist())

X = df[numeric_features].fillna(0)
y = df["is_declining_label"]
groups = df["client_id"]

print(f"Number of numeric features: {len(numeric_features)}")
print("\nNumeric features:")
print(numeric_features)

Number of numeric features: 29

Numeric features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


The first evaluation uses an 80/20 stratified random split. This provides the comparison point for the grouped evaluation.

In [65]:
# Standard random train/test split.
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y)

model_before = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1)

model_before.fit(X_train_random, y_train_random)

proba_before = model_before.predict_proba(X_test_random)[:, 1]
pred_before = model_before.predict(X_test_random)

metrics_before = {
    "split": "Random split (before)",
    "roc_auc": roc_auc_score(y_test_random, proba_before),
    "avg_precision": average_precision_score(
        y_test_random,
        proba_before
    ),
    "precision": precision_score(
        y_test_random,
        pred_before,
        zero_division=0
    ),
    "recall": recall_score(
        y_test_random,
        pred_before,
        zero_division=0)}

print("Random split results:")
for key, value in metrics_before.items():
    if key != "split":
        print(f"{key}: {value:.3f}")

Random split results:
roc_auc: 0.820
avg_precision: 0.830
precision: 0.719
recall: 0.859


The second evaluation uses five-fold GroupKFold validation grouped by client_id.

This prevents pages belonging to the same client from being distributed across both training and test sets within a fold. The goal is to measure how well the model generalizes to clients that were not represented during training.

In [66]:
# Five-fold grouped validation by client_id.
gkf = GroupKFold(n_splits=5)
fold_metrics = []
for fold, (train_idx, test_idx) in enumerate(
    gkf.split(X, y, groups=groups),
    start=1
):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1)

    model.fit(X_train, y_train)

    proba = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)

    fold_metrics.append({
        "fold": fold,
        "roc_auc": roc_auc_score(y_test, proba),
        "avg_precision": average_precision_score(
            y_test,
            proba
        ),
        "precision": precision_score(
            y_test,
            pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_test,
            pred,
            zero_division=0 )})

fold_df = pd.DataFrame(fold_metrics)

print("Grouped validation results:")
display(fold_df.round(3))

Grouped validation results:


,fold,roc_auc,avg_precision,precision,recall
0,1,0.711,0.703,0.623,0.699
1,2,0.740,0.830,0.709,0.952
2,3,0.851,0.760,0.570,0.951
3,4,0.752,0.790,0.762,0.928
4,5,0.799,0.812,0.739,0.916


In [67]:
metrics_after = {
    "split": "GroupKFold by client_id (after)",
    "roc_auc": fold_df["roc_auc"].mean(),
    "avg_precision": fold_df["avg_precision"].mean(),
    "precision": fold_df["precision"].mean(),
    "recall": fold_df["recall"].mean()}

print("Mean across client-grouped folds:")
for key, value in metrics_after.items():
    if key != "split":
        print(f"{key}: {value:.3f}")

Mean across client-grouped folds:
roc_auc: 0.771
avg_precision: 0.779
precision: 0.681
recall: 0.889


In [68]:
comparison = pd.DataFrame([
    metrics_before,
    metrics_after
]).set_index("split")

print("Validation comparison:")
display(comparison.round(3))

gap = (
    comparison.loc["Random split (before)"]
    - comparison.loc["GroupKFold by client_id (after)"]
)

print("\nRandom-split minus grouped-split gap:")
display(gap.round(3))

Validation comparison:


,roc_auc,avg_precision,precision,recall
split,,,,
Random split (before),0.820,0.830,0.719,0.859
GroupKFold by client_id (after),0.771,0.779,0.681,0.889



Random-split minus grouped-split gap:


,0
roc_auc,0.049
avg_precision,0.051
precision,0.039
recall,-0.031


The random-to-grouped split gap is measurable but does not represent a complete performance collapse. ROC-AUC decreases from 0.820 under the random split to 0.771 under client-grouped validation. Average precision decreases from 0.830 to 0.779, and precision decreases from 0.719 to 0.681.

Recall increases slightly from 0.859 to 0.889.

The lower ROC-AUC, average precision, and precision under client-grouped validation indicate that the random split likely benefited partly from client-specific patterns being shared between training and test data.

For this reason, the client-grouped results are the preferred performance estimate for reporting generalization across clients:

- ROC-AUC ≈ 0.77
- Average Precision ≈ 0.78
- Precision ≈ 0.68
- Recall ≈ 0.89

These metrics describe observed performance under the evaluation design; they do not prove that the model will perform identically on future clients.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [69]:
label_source_cols = [
    "trend_direction",
    "trend_pct"]

direct_leaks = [
    column
    for column in numeric_features
    if column in label_source_cols]

assert not direct_leaks, (
    f"Direct label leakage detected: {direct_leaks}")

print("PASS: No direct label-source columns are included in the features.")

PASS: No direct label-source columns are included in the features.


In [70]:
target_proxy_cols = [
    "impressions_last_30d",
    "impressions_prev_30d",
    "impressions_90d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d"]

proxy_features_present = [
    column
    for column in target_proxy_cols
    if column in numeric_features]

print("Potential target-proxy features:")

for column in proxy_features_present:
    print("-", column)

print(
    f"\n{len(proxy_features_present)} potential target-proxy "
    "features are present.")

Potential target-proxy features:
- impressions_last_30d
- impressions_prev_30d
- impressions_90d
- clicks_last_30d
- clicks_prev_30d
- sessions_last_30d
- sessions_prev_30d

7 potential target-proxy features are present.


In [71]:
correlations = (
    X.assign(label=y)
     .corr(numeric_only=True)["label"]
     .drop("label")
     .sort_values(key=abs, ascending=False))

print("Top feature-label correlations:")
display(correlations.head(10).round(3))

Top feature-label correlations:


,label
days_with_impressions,0.190
content_age_days,-0.164
age_tier_order,-0.156
word_count,0.119
char_count,0.108
impressions_last_30d,-0.094
days_since_last_update,0.081
clicks_last_30d,-0.072
sessions_last_30d,-0.064
ctr,-0.062


The correlations are modest rather than extremely high. However, correlation alone cannot rule out leakage because a feature can be predictive of a label without having a very large marginal correlation.

The more important concern is feature construction. Recent and previous-period impressions, clicks, and sessions are temporally close to the information used to define trend_direction. Therefore, these variables may function as near-proxies for the target.

This means the absence of the direct label column does not automatically establish that the feature set is fully independent of the target construction.

In [72]:
importance_series = (
    pd.Series(
        model_before.feature_importances_,
        index=numeric_features)
    .sort_values(ascending=False))

print("Top feature importances from the random-split model:")
display(importance_series.head(10).round(3))

top2_share = (
    importance_series.head(2).sum()
    / importance_series.sum())

print(
    f"\nTop-2 features account for {top2_share:.1%} "
    "of total feature importance.")

Top feature importances from the random-split model:


,0
impressions_prev_30d,0.354
impressions_90d,0.103
impressions_last_30d,0.090
days_with_impressions,0.086
avg_position,0.072
content_age_days,0.064
age_tier_order,0.036
clicks_last_30d,0.032
word_count,0.029
sessions_last_30d,0.025



Top-2 features account for 45.7% of total feature importance.


The top features include impressions_prev_30d, impressions_90d, and impressions_last_30d.

The top two features account for approximately 45.7% of total feature importance in the random-split model. This is substantial concentration, although it is lower than the approximately 86% concentration observed in the earlier ML-08 experiment.

Because these features describe recent traffic history and are closely related to the construction of trend_direction, their importance should not be interpreted as independent evidence that the model has discovered a new causal mechanism.

Instead, the result suggests that recent traffic patterns contain useful information about the observed decline label, while also creating a target-proxy concern.

Therefore:
- Direct leakage: not detected.
- Target-proxy concern: present.
- Interpretation: model performance should be treated as directional and decision-support evidence rather than proof of independent causal prediction.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [73]:
claims = [
    {
        "original": "The model predicts which pages will decline.",
        "rewritten": (
            "The model's score is associated with observed decline "
            "under the client-grouped evaluation, with a mean ROC-AUC "
            "of {:.2f} across five held-out client folds. It should be "
            "used as a decision-support signal for prioritizing review, "
            "not as a certain prediction."
        ).format(metrics_after["roc_auc"])
    },
    {
        "original": "Accuracy of 82.1% proves the model works well.",
        "rewritten": (
            "Under client-grouped evaluation, the model achieved "
            "{:.2f} precision and {:.2f} recall. Performance was lower "
            "on the grouped evaluation than on the random split, so "
            "the split gap should be disclosed when reporting results."
        ).format(
            metrics_after["precision"],
            metrics_after["recall"]
        )
    },
    {
        "original": "The model identifies the causes of ranking decay.",
        "rewritten": (
            "The model identifies features that are associated with "
            "the observed decline label. Because several traffic "
            "features are closely related to the label construction, "
            "the feature importance should be interpreted as "
            "associational rather than causal evidence."
        )
    }
]

for claim in claims:
    print("\nORIGINAL:")
    print(claim["original"])
    print("\nREWRITTEN:")
    print(claim["rewritten"])


ORIGINAL:
The model predicts which pages will decline.

REWRITTEN:
The model's score is associated with observed decline under the client-grouped evaluation, with a mean ROC-AUC of 0.77 across five held-out client folds. It should be used as a decision-support signal for prioritizing review, not as a certain prediction.

ORIGINAL:
Accuracy of 82.1% proves the model works well.

REWRITTEN:
Under client-grouped evaluation, the model achieved 0.68 precision and 0.89 recall. Performance was lower on the grouped evaluation than on the random split, so the split gap should be disclosed when reporting results.

ORIGINAL:
The model identifies the causes of ranking decay.

REWRITTEN:
The model identifies features that are associated with the observed decline label. Because several traffic features are closely related to the label construction, the feature importance should be interpreted as associational rather than causal evidence.


The validation audit shows that the model retains useful directional signal when evaluated on unseen clients, but performance is lower under client-grouped validation than under a standard random split.

The approximately 0.05 ROC-AUC difference indicates that the random split provided a somewhat more optimistic estimate of generalization.

The leakage audit found no direct inclusion of the label-source columns. However, several recent and previous-period traffic features are closely related to the construction of the trend_direction label. These features should therefore be treated as potential target proxies.

The strongest conclusion supported by this notebook is that the model provides a directional decision-support signal for prioritizing pages for review. The results do not establish causal relationships or guarantee that individual pages will decline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.